In [46]:
!uv add 'markitdown[pdf]'

Resolved 137 packages in 20ms
Checked 133 packages in 63ms


In [47]:
from markitdown import MarkItDown
from pathlib import Path

md = MarkItDown()

books_dir = Path("books")
output_dir = Path("book_text")
output_dir.mkdir(exist_ok=True)

for each_file in books_dir.glob("*.pdf"):
    print("Converting", each_file)

    result = md.convert(str(each_file))
    text = result.text_content

    output_path = output_dir / (each_file.stem + ".md")
    output_path.write_text(text, encoding="utf-8")

Converting books/think-c.pdf
Converting books/think-python-2e.pdf
Converting books/think-dsp.pdf
Converting books/physical-modeling-in-matlab.pdf
Converting books/think-os.pdf
Converting books/think-java-2e.pdf


KeyboardInterrupt: 

In [ ]:
text = (output_dir / "think-python-2e.md").read_text(encoding="utf-8")
print(len(text.splitlines()))

9703


In [ ]:
! uv add gitsource
from gitsource import chunk_documents

# output_dir = Path("book_text")
# documents = []

# for md_file in output_dir.glob("*.md"):
#     text = md_file.read_text(encoding="utf-8")
#     lines = text.splitlines()

#     non_empty_lines = []
#     for each_line in lines:
#         if each_line.strip():
#             non_empty_lines.append(each_line)

#     document = {
#         "source": md_file.name,
#         "content": non_empty_lines
#     }
#     documents.append(document)

# print(len(documents))

from pathlib import Path
from gitsource import chunk_documents

output_dir = Path("books_text")
documents = []

for md_file in output_dir.glob("*.md"):
    lines = md_file.read_text(encoding="utf-8").splitlines()
    documents.append({
        "source": md_file.name,
        "content": [line for line in lines if line.strip()],
    })

think_python = [d for d in documents if d["source"] == "think-python-2e.md"]

chunks = chunk_documents(think_python, size=100, step=50)
print(len(chunks))


Resolved 137 packages in 1ms
Checked 133 packages in 3ms


189


In [ ]:
len(chunks)

189

In [ ]:
from minsearch import Index

all_chunks = chunk_documents(documents, size=100, step=50)

def prepare_documents(chunks):
    prepared = []
    for chunk in chunks:
        doc = chunk.copy()
        content = doc["content"]
        if isinstance(content, list):
            doc["content"] = "\n".join(content)
        prepared.append(doc)
    return prepared

index_documents = prepare_documents(all_chunks)

index = Index(text_fields=["content"])
index.fit(index_documents)

print(len(index_documents))

988


In [ ]:
results = index.search("python function definition", num_results=5)

In [ ]:
results

[{'start': 1800,
  'content': "| ------ | ------------ | --- | --- |\n| total  | = total +    | a;  |     |\nend\nans = total\nIfyouwereusinganyofthosevariablenamesbeforecallingthisscript, youmightbesurprised\nto find, after running the script, that their values had changed. If you have two scripts that\n48 Functions\nuse the same variable names, you might find that they work separately and then break when\nyou try to combine them. This kind of interaction is called a collision.\nname\nAs the number of scripts you write increases, and they get longer and more complex, name\ncollisions become more of a problem. Avoiding this problem is one of several motivations for\nfunctions.\n| 5.2 | Defining | Functions |     |     |\n| --- | -------- | --------- | --- | --- |\nA function is like a script, except that each function has its own workspace, so any variables\ndefined inside a function only exist while the function is running and don’t interfere with\nvariables in other workspaces, even 

In [48]:
import json

from openai import OpenAI
openai_client = OpenAI()

search_results = json.dumps(results, indent=2)
query = 'python function definition'




In [52]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_results}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        query=question,
        search_results=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [53]:
answer = rag('python function definition')
print(answer)

In Python, a function is defined using the `def` keyword followed by the function name and parentheses containing any parameters. After the colon, you write the body of the function, which may consist of one or more statements. Here’s a simple example of a function definition in Python:

```python
def my_function(x):
    s = sin(x)  # Using sin from the math module
    c = cos(x)  # Using cos from the math module
    res = abs(s) + abs(c)
    return res
```

### Key Points:

1. **Function Signature**: The first line (`def my_function(x):`) is the signature of the function, indicating its name and parameters.
2. **Function Body**: This includes the operations performed and the calculations made based on the inputs.
3. **Return Statement**: The `return` keyword is used to send back a result from the function.

### Example Call:
After defining the function, you can call it like this:

```python
result = my_function(1)
print(result)  # Output the result
```

This will execute the function 

In [54]:
def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    usage = response.usage
    return response.output_text, usage.input_tokens, usage.output_tokens

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, input_tokens, output_tokens = llm(prompt, instructions)
    return answer, input_tokens, output_tokens

In [56]:
answer, input_tokens, output_tokens = rag('python function definition')
print(answer)
print("input tokens:", input_tokens)
print("output tokens:", output_tokens)

In Python, a function is defined using the `def` keyword, followed by the function's name and parentheses that may include parameters. Below is a simple structure to define a function in Python:

```python
def my_function(parameter1, parameter2):
    # Function body: operations to be performed
    result = parameter1 + parameter2
    return result
```

### Key Components:
- **Function Definition**: Starts with `def`, followed by the function name.
- **Parameters**: Variables that the function takes as input.
- **Function Body**: The code block where operations are performed.
- **Return Statement**: The output of the function is specified with the `return` statement.

### Example:
Here's a complete example of a function that calculates the hypotenuse of a right triangle:

```python
def hypotenuse(a, b):
    return (a**2 + b**2)**0.5
```

You can call this function by passing the lengths of the two sides:

```python
result = hypotenuse(3, 4)
print(result)  # Output will be 5.0
```

This 

In [58]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")



In [59]:
def llm_structured(user_prompt, instructions, output_type, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    usage = response.usage
    return response.output_parsed, usage.input_tokens, usage.output_tokens

def rag_structured(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, input_tokens, output_tokens = llm_structured(prompt, instructions, RAGResponse)
    return answer, input_tokens, output_tokens

In [60]:
q = 'python function definition'

_, plain_in, plain_out = rag(q)
structured, struct_in, struct_out = rag_structured(q)

print("unstructured input tokens:", plain_in)
print("structured input tokens:", struct_in)
print("difference:", struct_in - plain_in)

print(structured)

unstructured input tokens: 7928
structured input tokens: 8114
difference: 186
answer="### Python Function Definition\n\nIn Python, a function is defined using the `def` keyword followed by the function name and parentheses containing any parameters. Here’s a basic structure of a function:\n\n```python\ndef function_name(parameters):\n    # Function body\n    # Perform actions\n    return result   # Optional\n```\n\n### Example:\nHere's an example of a function that adds two numbers:\n\n```python\ndef add_numbers(a, b):  \n    result = a + b  \n    return result  \n```\n\nYou would call this function by using its name followed by the arguments in parentheses:\n```python\nsum = add_numbers(3, 5)  # sum will be 8\n```\n\nThis definition allows for encapsulating reusable code, where the function can be called multiple times with different arguments without polluting the main workspace with variables defined inside the function." found_answer=True confidence=0.95 confidence_explanation='The